# Predictive Employee Performance Analytics using Random Forest
### Topic: HR Analytics and Workforce Optimization

This Jupyter Notebook contains the complete machine learning pipeline to analyze employee productivity and predict their performance scores using the **Random Forest Classifier** algorithm. It is designed to run seamlessly in **Google Colab**.

### Objectives:
1. **Exploratory Data Analysis (EDA)**: Understand the distribution of employee characteristics and identify key drivers of performance.
2. **Data Preprocessing**: Encode categorical features, clean missing data, and prepare features for model training.
3. **Model Development**: Train a Random Forest Classifier to predict the `Performance_Score` of employees.
4. **Evaluation**: Measure accuracy, precision, recall, F1-score, and plot the confusion matrix and feature importances.
5. **Export & Download**: Save the trained model and encoders to a single file and download it to use in a Streamlit application.

## Step 1: Import Libraries and Mount Google Drive / Upload File
We will first import all the required scientific computing, visualization, and machine learning libraries.

In [ ]:
# Standard library imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Configure visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Libraries imported successfully!")

### Uploading Dataset in Google Colab
Run the following cell to upload your `Extended_Employee_Performance_and_Productivity_Data.csv` file directly to the Colab session directory.

In [ ]:
# File upload utility for Google Colab
try:
    from google.colab import files
    print("Please upload 'Extended_Employee_Performance_and_Productivity_Data.csv' below:")
    uploaded = files.upload()
    dataset_path = 'Extended_Employee_Performance_and_Productivity_Data.csv'
except ImportError:
    # Fallback to local path if running outside Google Colab
    dataset_path = 'Extended_Employee_Performance_and_Productivity_Data.csv'
    print(f"Not running in Google Colab. Using default path: '{dataset_path}'")

## Step 2: Load and Inspect the Dataset

In [ ]:
# Load data
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded successfully! Dimensions: {df.shape[0]} rows, {df.shape[1]} columns.")
else:
    print(f"[ERROR] File '{dataset_path}' not found. Please upload it and run the cell again.")

In [ ]:
# Display dataset info
df.info()

In [ ]:
# Display first 5 rows
df.head()

## Step 3: Exploratory Data Analysis (EDA)
Let's visualize the target variable and its relationships with other key variables to gain initial insights.

In [ ]:
# 1. Performance Score Distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='Performance_Score', data=df, palette='viridis')
plt.title('Distribution of Employee Performance Scores', fontsize=14, fontweight='bold')
plt.xlabel('Performance Score', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

In [ ]:
# 2. Performance Score vs Monthly Salary
plt.figure(figsize=(10, 6))
sns.boxplot(x='Performance_Score', y='Monthly_Salary', data=df, palette='Set2')
plt.title('Monthly Salary Distribution across Performance Scores', fontsize=14, fontweight='bold')
plt.xlabel('Performance Score', fontsize=12)
plt.ylabel('Monthly Salary ($)', fontsize=12)
plt.show()

In [ ]:
# 3. Employee Satisfaction Score vs Performance Score
plt.figure(figsize=(10, 6))
sns.violinplot(x='Performance_Score', y='Employee_Satisfaction_Score', data=df, palette='coolwarm')
plt.title('Employee Satisfaction by Performance Score', fontsize=14, fontweight='bold')
plt.xlabel('Performance Score', fontsize=12)
plt.ylabel('Employee Satisfaction Score (1-5)', fontsize=12)
plt.show()

In [ ]:
# 4. Correlation Heatmap (numerical variables only)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Employee_ID' in num_cols:
    num_cols.remove('Employee_ID')

plt.figure(figsize=(12, 10))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing and Feature Engineering
To train our Random Forest model, we must:
1. Drop high-cardinality/uninformative identifier columns (`Employee_ID`, `Hire_Date`).
2. Encode categorical columns using label encoders.
3. Convert boolean variables (like `Resigned`) into integer formats (0/1).

In [ ]:
# 1. Drop irrelevant columns
df_clean = df.drop(columns=['Employee_ID', 'Hire_Date'])

# 2. Convert boolean Resigned column to int
df_clean['Resigned'] = df_clean['Resigned'].astype(int)

# 3. Encode categorical columns
cat_cols = ['Department', 'Gender', 'Job_Title', 'Education_Level']
encoders = {}
categorical_mappings = {}

for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    encoders[col] = le
    categorical_mappings[col] = list(le.classes_)
    print(f"Encoded column '{col}': Mapping classes -> {categorical_mappings[col]}")

## Step 5: Model Training (Random Forest Classifier)
We split the preprocessed data into features (X) and target (y), perform a train-test split, and fit a Random Forest model.

In [ ]:
# Split into X and y
X = df_clean.drop(columns=['Performance_Score'])
y = df_clean['Performance_Score']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Initialize Random Forest Classifier
# We set max_depth to control model complexity and avoid excessive file sizes
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=12, 
    random_state=42, 
    n_jobs=-1
)

print("Training the Random Forest model...")
rf_model.fit(X_train, y_train)
print("Model training complete!")

## Step 6: Model Evaluation
Let's review the model's accuracy, classification metrics, and plot the confusion matrix and feature importances.

In [ ]:
# Predict on test set
y_pred = rf_model.predict(X_test)

# Print overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Classification Accuracy: {accuracy * 100:.2f}%")

# Print detailed classification report
print("
Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Rating 1', 'Rating 2', 'Rating 3', 'Rating 4', 'Rating 5'],
            yticklabels=['Rating 1', 'Rating 2', 'Rating 3', 'Rating 4', 'Rating 5'])
plt.title('Confusion Matrix - Random Forest Predictor', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Ratings', fontsize=12)
plt.ylabel('Actual Ratings', fontsize=12)
plt.show()

In [ ]:
# Plot Feature Importances
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feat_imp.plot(kind='barh', color='skyblue')
plt.title('Feature Importances in Predicting Employee Performance', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance Value', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()
plt.show()

## Step 7: Export Model and Encoders for Streamlit
We bundle the trained model, feature columns, and category label mappings in a dictionary and dump them to `performance_model.pkl`. We then use Colab's built-in file utility to trigger a download of the saved model.

In [ ]:
# Package model and metadata
model_data = {
    'model': rf_model,
    'features': list(X.columns),
    'categorical_mappings': categorical_mappings
}

# Save model locally in the Colab disk
output_filename = 'performance_model.pkl'
joblib.dump(model_data, output_filename)
print(f"Model successfully saved as '{output_filename}'!")

In [ ]:
# Trigger browser download of the model file from Google Colab
try:
    from google.colab import files
    print("Initiating browser download of the model file...")
    files.download(output_filename)
except ImportError:
    print("Not running in Google Colab. The model file remains saved locally on disk.")